# Fórmulas en R para Modelos de Regresión

A continuación se presenta una guía sobre cómo escribir fórmulas de modelos de regresión en R, incluyendo el manejo de variables categóricas y la aplicación de funciones a las variables predictoras.

## Datos de ejemplo


Supongamos que tenemos la siguiente base de datos con 4 variables continuas $X_{1},X_{2},X_{3},X_{4}$, una variable nominal $catNum$ de dos niveles ($0, 1$) y una variable nominal $catLetras$ de tres niveles ($A,B,C$).




In [1]:
library(tidyverse)
set.seed(42)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:

n <- 100

# regresoras
variable1 <- runif(n, 0, 100)
variable2 <- runif(n, 0, 50)
variable3 <- runif(n, 0, 30)
variable4 <- runif(n, 0, 10)

# respuesta
respuesta <- 3 + 0.5 * variable1 + 2 * variable2 - 1.5 * variable3 +
  rnorm(n, 0, 5) - 0.2 * variable4


#categoricas
categoriasLetras <- c("A", "B", "C")
catLetras <- sample(categoriasLetras, n, replace = TRUE)

categoriasNum <- c(0, 1)
catNum <- sample(categoriasNum, n, replace = TRUE)

#datos
df <- data.frame(
  X1 = variable1,
  X2 = variable2,
  X3 = variable3,
  X4 = variable4,
  Y = respuesta,
  catLetras = factor(catLetras, levels = categoriasLetras),
  catNum = factor(catNum, levels = categoriasNum)
)

head(df)


,X1,X2,X3,X4,Y,catLetras,catNum
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>
1,64.17455,47.1227846,4.736403,8.784290,126.32801,C,0
2,51.90959,48.1304007,13.269739,9.306049,113.74748,A,1
3,73.65883,36.9927640,29.032010,3.921785,62.59826,C,1
4,13.46666,36.6622953,14.537638,1.588468,55.17949,C,0
5,65.69923,26.7880645,7.573753,3.199476,73.89611,C,1
6,70.50648,0.1136483,7.790699,3.069656,20.91028,A,1


## Modelo matemático y sintaxis en R

Supongamos que queremos ajustar el modelo de regresión simple:

$ y = \beta_{0} + \beta_{1}x_{1} $

La "fórmula" del modelo en R se escribe como:

**Y ~ X1**

Es decir, la fórmula del modelo se escribe **usando los nombres de las columnas del df**. La variable a la izquierda de "~" es la variable dependiente $y$ y la variable de la derecha es la independiente $x_{1}$. Si quisieramos agregar mas variables basta con agregar "+" antes de cada variable (Y ~ X1 + X2).

Ademas, no es necesario especificar el intercepto $\beta_{0}$, ya que por default se agrega, en caso de querer quitarlo se agrega "-1" a la expresion.
**Quitar el intercepto en la formula es un error**.

Por ejemplo:

$ y =\beta_{2}x_{2} $

**Y ~ X2 - 1**


# Ajuste de un modelo de regresión

Para ajustar un modelo de regresión en R, se utiliza la función `lm()`, que significa "linear model" (modelo lineal). La sintaxis básica para ajustar un modelo de regresión lineal es la siguiente:

```R
modelo <- lm(formula, data)
```
A continuación, se describen algunos ejemplos de cómo ajustar modelos de regresión con variables de diferentes escalas/formatos.


## Variables numéricas

Supongamos que es de interes ajustar $ y = \beta_{0} + \beta_{1}x_{1} $

La "fórmula" del modelo de regresión que se ajusta:

**Y ~ X1**

Y se implementa:


In [4]:
modelo <- lm(Y ~ X1 , data = df)


confint(modelo)

summary(modelo)


,2.5 %,97.5 %
(Intercept),24.7378698,51.5606843
X1,0.2317926,0.6872361



Call:
lm(formula = Y ~ X1, data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-64.672 -20.991   1.937  25.281  64.686 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  38.1493     6.7582   5.645 1.61e-07 ***
X1            0.4595     0.1148   4.004 0.000121 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 33.94 on 98 degrees of freedom
Multiple R-squared:  0.1406,	Adjusted R-squared:  0.1318 
F-statistic: 16.04 on 1 and 98 DF,  p-value: 0.000121


## Variables categóricas en el modelo de regresión

____
**Observación:**

Cuando tenemos una variable categórica de *k niveles*, **en el preprocesamiento se debe indicar que es categóorica** asi como el orden de los niveles (categorías) aún cuando sea nominal y los niveles no tienen un orden dado. Realizaremos esto con `factor()` luego de haber creado el dataframe con las variables.

Al indicar el orden de los niveles, a la categoría que se encuentre en primer lugar le llamaremos **nivel de referencia** de la variable. Naturalmente si excluimos el nivel de referencia nos quedan k-1 categorías.
____

El correcto tratamiento de las variables categóricas en los modelos de regresión es crucial, pues su manejo es un aspecto que se suele omitir dentro de la teoría pero es común encontrarnos con datos que contengan variables de esta naturaleza.

Al ajustar un modelo con `lm()`, si incluimos una variable categórica con k categorías, en automático se realiza un preprocesamiento al excluir al nivel de referencia, y descomponiendo esta variable en k-1 variables binarias que servirán para indicar si una observación corresponde a cierta categoria (1) o no corresponde a esa categoría (0), esto sin importar si es nominal u ordinal.

Por ejemplo, supongamos que incluimos la variable _Raza_, que tiene **3** categorías: Blanca, Negra y Latina, y definimos los niveles con el siguiente orden: (1)Latina, (2)Blanca, (3)Negra.

El preprocesamiento que realiza `lm()` es crear **2** variables indicadoras (0,1) para las categorias que no sean el **nivel de referencia** (Blanca y Negra), y a cada una de estas nuevas variables estimar un coeficiente $\beta_{r}$

Como se puede notar, **es muy importante tener claro cuál es el nivel de referencia** de la variable categórica para evitar confusiones con los coeficientes, si cometemos la mala práctica de que en el preprocesamiento no indicamos que nuestra variable es categórica y por ende no especificamos el nivel de referencia, R lo elige por nosotros.

A continuación se muestra este procedimiento de formal manual, para ilustrar mejor el ejemplo. Notemos que pasamos de la variable _Raza_ a las variables binarias _Raza Blanca_ y _Raza Negra_, podemos interpretar que si en ambas variables hay un 0, significa que la observacion corresponde al nivel de referencia de la categoria (Latina).


In [ ]:
razas <- c("Blanca", "Negra", "Latina")


df$Raza <- factor(sample(razas, n, replace = TRUE),
                  levels = c("Latina", "Blanca", "Negra"))


raza_dummies <- model.matrix(~ Raza, data = df)

df_with_dummies <- cbind(df["Raza"], raza_dummies[, -1])

head(df_with_dummies)


,Raza,RazaBlanca,RazaNegra
,<fct>,<dbl>,<dbl>
1,Blanca,1,0
2,Negra,0,1
3,Latina,0,0
4,Blanca,1,0
5,Blanca,1,0
6,Latina,0,0


### La variable categórica está en formato de string

Supongamos que nos interesa ajustar

$y = \beta_{0} + \beta_{2}$**catLetras**  


Notemos que **catLetras** son strings ("A","B","C")

Entonces el modelo se ajusta sin ninguna modificación, debido a que la funciónn identifica que la variable es categórica al estar en formato string (aún cuando no se haya indicado en el preprocesamiento).

In [6]:
modeloCatLetras <- lm(Y ~ catLetras, data = df)
summary(modeloCatLetras)


Call:
lm(formula = Y ~ catLetras, data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-76.300 -25.180   0.539  27.241  74.326 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)   63.372      6.499   9.751 4.63e-16 ***
catLetrasB    -4.053      9.426  -0.430    0.668    
catLetrasC    -1.654      8.769  -0.189    0.851    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 36.76 on 97 degrees of freedom
Multiple R-squared:  0.001916,	Adjusted R-squared:  -0.01866 
F-statistic: 0.09312 on 2 and 97 DF,  p-value: 0.9112


Notemos que catLetras**B** es la variable binaria que corresponde a la categoría B de la variable catLetras, y catLetras**C** a la categoría C. Ambas tienen su respectivo coeficiente.

Surge la pregunta, **¿Qué coeficiente está asociado al nivel de referencia (la categoría A)? ¿Perdimos la información que esa categoría nos daba?**

**La respuesta es que el "efecto" de la categoría A fue agregado al valor del intercepto $\beta_{0}$**.

### La variable categórica está en formato de número

Supongamos que nos interesa ajustar

$y = \beta_{0}  + \beta_{2}$**catNum**  


Notemos que **catNum** está representada por números (0,1) aunque sea categórica, y ademas cometimos la mala práctica de no indicarlo en el preprocesamiento.

Hay dos alternativas, la primera es especificar que es categórica en el preprocesamiento, la segunda es ajustar el modelo **especificando en la fórmula** que **catNum** es categórica usando `factor()`. Sin embargo esto último no es recomendable, ya que el preprocesamiento debe realizarse antes de ajustar el modelo.

In [7]:
modeloCatNum <- lm(Y ~ factor(catNum), data = df)
summary(modeloCatNum)


Call:
lm(formula = Y ~ factor(catNum), data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-78.724 -25.619   0.424  28.041  72.481 

Coefficients:
                Estimate Std. Error t value Pr(>|t|)    
(Intercept)      61.1639     5.6491   10.83   <2e-16 ***
factor(catNum)1   0.6684     7.4176    0.09    0.928    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 36.61 on 98 degrees of freedom
Multiple R-squared:  8.285e-05,	Adjusted R-squared:  -0.01012 
F-statistic: 0.00812 on 1 and 98 DF,  p-value: 0.9284


catNum**1** es la variable binaria correspondiente a la categoría 1 de la variable catNum y tiene su respectivo coeficiente. El efecto del nivel de referencia (la categoria 0) se ve reflejado en el valor del intercepto $\beta_{0}$

## Transformaciones de las variables


### Funciones comunes

Supongamos que es de interes ajustar $ y = \beta_{0} + \beta_{1}log(x_{1})$

La "fórmula" del modelo en R se escribe como:

**Y ~ ln(X1)**

Y se implementa:


In [6]:
modelo_log <- lm(Y ~ log(X1), data = df)
summary(modelo_log)




Call:
lm(formula = Y ~ log(X1), data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-75.868 -23.958   1.549  28.064  68.565 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)   41.870      9.755   4.292 4.16e-05 ***
log(X1)        5.610      2.587   2.168   0.0325 *  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 35.76 on 98 degrees of freedom
Multiple R-squared:  0.04579,	Adjusted R-squared:  0.03605 
F-statistic: 4.702 on 1 and 98 DF,  p-value: 0.03254


Se puede usar otras funciones comunes como `sqrt()`, `exp()`, `I()` (para potencias), entre otras.

In [7]:
(lm(Y ~ sqrt(X2), data = df))

(lm(Y ~ exp(X3), data = df))    

(lm(Y ~ I(X4^2), data = df))


Call:
lm(formula = Y ~ sqrt(X2), data = df)

Coefficients:
(Intercept)     sqrt(X2)  
     -24.71        17.63  



Call:
lm(formula = Y ~ exp(X3), data = df)

Coefficients:
(Intercept)      exp(X3)  
  6.327e+01   -5.647e-12  



Call:
lm(formula = Y ~ I(X4^2), data = df)

Coefficients:
(Intercept)      I(X4^2)  
   64.45660     -0.08458  
